In [1]:
!pip install ultralytics opencv-python #ulta-yolo and open cv- vidoe precessin

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.1 MB/s eta 0:00:00


step1

Input Video
    
YOLO
    

Detect Ships, BOUNDARY BOX AND CONFIDENCE SCORE
    

Save Detection Video

In [2]:
import cv2  #reading video,box draw, writing vdeo
from ultralytics import YOLO  #image object detect

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
model = YOLO("yolo12s.pt")

[
    {
        class: ship,
        confidence: 0.94,
        bbox:[100,50,300,200]
    }
]

In [4]:
input_video = "/content/video7.mp4"
output_video = "step1_detection.mp4"

In [5]:
cap = cv2.VideoCapture(input_video)  #input video

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

writer = cv2.VideoWriter(    #for output video size width same as input fps and resolution
    output_video,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

In [6]:
frame_count = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame, imgsz=1536)  #input is frames Resizes the frame to 1536×1536

    for result in results:

        boxes = result.boxes

        for box in boxes:  #all object check

            x1, y1, x2, y2 = map(int, box.xyxy[0]) #coordinants

            conf = float(box.conf[0])  #confidence

            cls = int(box.cls[0])  #class

            label = model.names[cls]  #Number to Name,class 8 to boat

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0,255,0),
                2
            )

            cv2.putText(
                frame,
                f"{label} {conf:.2f}",   #display
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0,255,0),  #green
                2 #thickness
            )

    writer.write(frame)  #save frames

    frame_count += 1  #add frames

print(f"Processed {frame_count} frames")


0: 864x1536 2 persons, 1 boat, 108.4ms
Speed: 28.8ms preprocess, 108.4ms inference, 42.1ms postprocess per image at shape (1, 3, 864, 1536)

0: 864x1536 2 persons, 1 boat, 53.6ms
Speed: 20.8ms preprocess, 53.6ms inference, 3.9ms postprocess per image at shape (1, 3, 864, 1536)

0: 864x1536 2 persons, 1 boat, 53.6ms
Speed: 16.0ms preprocess, 53.6ms inference, 1.7ms postprocess per image at shape (1, 3, 864, 1536)

0: 864x1536 2 persons, 1 boat, 53.7ms
Speed: 14.7ms preprocess, 53.7ms inference, 3.3ms postprocess per image at shape (1, 3, 864, 1536)

0: 864x1536 2 persons, 1 boat, 53.6ms
Speed: 15.1ms preprocess, 53.6ms inference, 1.6ms postprocess per image at shape (1, 3, 864, 1536)

0: 864x1536 1 person, 1 boat, 53.6ms
Speed: 15.1ms preprocess, 53.6ms inference, 1.4ms postprocess per image at shape (1, 3, 864, 1536)

0: 864x1536 2 persons, 1 boat, 53.3ms
Speed: 15.7ms preprocess, 53.3ms inference, 2.4ms postprocess per image at shape (1, 3, 864, 1536)

0: 864x1536 2 persons, 1 boat, 

KeyboardInterrupt: 

In [ ]:
cap.release()
writer.release()

print("Detection video saved")

In [ ]:
import os

print(os.path.exists("/content/step1_detection.mp4"))
print(os.path.getsize("/content/step1_detection.mp4"))

In [ ]:
from IPython.display import Video

Video("step1_detection.mp4", embed=True)

STEP2

Integrating bytetrack to assign a unique id to all

In [7]:
input_video = "/content/video7.mp4"
output_video = "step2_tracking4.mp4"

In [8]:
model = YOLO("yolo12s.pt")

In [9]:
import cv2
import os

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise ValueError("Could not open input video")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

In [12]:
%%writefile botsort_custom.yaml
 #my own bot sort for better quality and longer track buffer

tracker_type: botsort

track_high_thresh: 0.45
track_low_thresh: 0.10
new_track_thresh: 0.45

track_buffer: 120

match_thresh: 0.75
fuse_score: True

gmc_method: sparseOptFlow

proximity_thresh: 0.5
appearance_thresh: 0.25
with_reid: False
model: ''  # Added dummy model entry to prevent AttributeError

Writing botsort_custom.yaml


In [13]:
!cat botsort_custom.yaml

 #my own bot sort for better quality and longer track buffer

tracker_type: botsort

track_high_thresh: 0.45
track_low_thresh: 0.10
new_track_thresh: 0.45

track_buffer: 120

match_thresh: 0.75
fuse_score: True

gmc_method: sparseOptFlow

proximity_thresh: 0.5
appearance_thresh: 0.25
with_reid: False
model: ''  # Added dummy model entry to prevent AttributeError


In [14]:
import cv2
import os

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise ValueError(f"Could not open input video: {input_video}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

frame_count = 0
unique_ids = set()

MIN_AREA = 1500   # remove tiny noisy detections

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model.track(
        frame,
        imgsz=1536,
        conf=0.55,
        persist=True,  #keep memory from previous frame
        tracker="botsort_custom.yaml",
        verbose=False
    )

    for result in results:

        if result.boxes is None:
            continue

        for box in result.boxes:

            if box.id is None:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            area = (x2 - x1) * (y2 - y1)

            if area < MIN_AREA:
                continue

            conf = float(box.conf[0])

            track_id = int(box.id[0])

            unique_ids.add(track_id)

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"Ship {conf:.2f} | ID:{track_id}",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    writer.write(frame)
    frame_count += 1

cap.release()
writer.release()

print(f"Processed {frame_count} frames")
print(f"Total unique track IDs: {len(unique_ids)}")

if os.path.exists(output_video):
    print(f"Output size: {os.path.getsize(output_video)/(1024*1024):.2f} MB")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 283ms
Prepared 1 package in 64ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Processed 1737 frames
Total unique track IDs: 6
Output size: 430.60 MB


In [15]:
cap.release()
writer.release()

print("Tracking video saved")

Tracking video saved


In [17]:
import os
print(os.path.getsize("step2_tracking4.mp4"))

451519961
